# RIASEC27 — Công đoạn 01: Làm sạch và kiểm tra chất lượng dữ liệu
## 1. Mục tiêu
Kiểm tra cấu trúc, ghép một-một hai bảng khảo sát, xác minh kiểu/miền Likert và chuẩn bị dữ liệu sạch trong bộ nhớ. Giữ đủ 4.910 bản ghi khi chưa có lỗi xác định rõ; không bắt buộc xóa dữ liệu để gọi là làm sạch. Không chia tập, chuẩn hóa, chọn đặc trưng hoặc huấn luyện.

## 2. Nguồn dữ liệu và cơ sở xử lý
Chỉ phân tích `dataLabanAI.xlsx`, đọc RIASEC_27, SO_THICH_6, CODEBOOK_33, THONG_TIN. Không dùng dữ liệu F54. Bản gốc `Data Student_Full.xlsx` đã được đối chiếu tại Notebook 00; không dùng thay nguồn ở đây. Notebook tự đọc workbook, không cần biến hay file dữ liệu sạch của notebook khác.

Likert 1–5 là thang thứ bậc; phương sai mã số là mô tả kỹ thuật, không chứng minh khoảng cách tâm lý bằng nhau hoặc tính hợp lệ của người trả lời. Không có nhãn 33 ngành đại học. Q5 là sáu mức sở thích tự báo cáo được hỏi riêng, không phải nhãn ngành hoặc điểm RIASEC tính từ X27.


In [1]:
from pathlib import Path
import sys,hashlib
import pandas as pd
import numpy as np
from IPython.display import display
assert sys.version_info[:2] == (3,14)
ROOT = next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'dataLabanAI.xlsx').is_file())
workbook_path = ROOT/'dataLabanAI.xlsx'
sha_before = hashlib.sha256(workbook_path.read_bytes()).hexdigest()
sha_notebook00 = 'c8a9445edba02a37cf862a3b1eabffba910eb7eeaf1d48b4c6f8ac19fea296c8'
assert sha_before == sha_notebook00, 'Workbook khác phiên bản Notebook 00; cần audit lại trước xử lý.'
excel = pd.ExcelFile(workbook_path)
assert excel.sheet_names == ['THONG_TIN','RIASEC_27','SO_THICH_6','CODEBOOK_33','KIEM_TRA']
raw_riasec = pd.read_excel(excel,sheet_name='RIASEC_27')
raw_q5 = pd.read_excel(excel,sheet_name='SO_THICH_6')
raw_codebook = pd.read_excel(excel,sheet_name='CODEBOOK_33')
raw_info = pd.read_excel(excel,sheet_name='THONG_TIN')
# Bản sao gốc dùng đối chiếu, không thay đổi các DataFrame này.
original_riasec = raw_riasec.copy(deep=True)
original_q5 = raw_q5.copy(deep=True)
original_codebook = raw_codebook.copy(deep=True)
original_info = raw_info.copy(deep=True)
print('Python:',sys.version,'; Interpreter:',sys.executable)
print('SHA-256:',sha_before)
display(pd.DataFrame([{'Sheet':name,'Số dòng':len(frame),'Số cột':frame.shape[1]}
    for name,frame in [('RIASEC_27',raw_riasec),('SO_THICH_6',raw_q5),('CODEBOOK_33',raw_codebook),('THONG_TIN',raw_info)]]))


Python: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)] ; Interpreter: C:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe
SHA-256: c8a9445edba02a37cf862a3b1eabffba910eb7eeaf1d48b4c6f8ac19fea296c8
         Sheet  Số dòng  Số cột
0    RIASEC_27     4910      28
1   SO_THICH_6     4910       7
2  CODEBOOK_33       33       5
3    THONG_TIN       23       2


## 3. Kiểm tra cấu trúc và ghép một-một
Giữ thứ tự 27 câu đã xác nhận tại Notebook 00. Ghép outer có indicator và validate="one_to_one" để phát hiện dòng không khớp; chỉ tiếp tục khi không tăng/mất dòng. SOURCE_ROW là tham chiếu hàng nguồn, không hiển thị giá trị từng dòng và không dùng làm feature.

In [2]:
question_columns = ['R1','I1','A1','R2','I2','A2','S1','S2','R3','A3','S3','E1','R4','S4','E2','R5','I3','A4','E3','I4','A5','E4','C1','E5','C2','C3','C4']
q5_columns = ['Q5_LikeRe','Q5_LikeIn','Q5_LikeAr','Q5_LikeSo','Q5_LikeEn','Q5_LikeCo']
survey_columns = question_columns+q5_columns
assert raw_riasec.shape == (4910,28) and raw_q5.shape == (4910,7)
assert list(raw_riasec.columns) == ['SOURCE_ROW']+question_columns
assert list(raw_q5.columns) == ['SOURCE_ROW']+q5_columns
assert raw_codebook.MA_COT.is_unique and raw_codebook.MA_COT.tolist() == survey_columns
assert raw_codebook.iloc[:27]['LOAI'].eq('RIASEC rút gọn').all()
assert raw_codebook.iloc[27:]['LOAI'].eq('Mức ưa thích nhóm nghề').all()
assert raw_codebook.iloc[:27]['THANG_DIEM_GOC'].nunique()==1
assert raw_codebook.iloc[27:]['THANG_DIEM_GOC'].nunique()==1
for frame in [raw_riasec,raw_q5]:
    assert frame.SOURCE_ROW.notna().all() and frame.SOURCE_ROW.is_unique
    assert pd.api.types.is_integer_dtype(frame.SOURCE_ROW)
assert set(raw_riasec.SOURCE_ROW) == set(raw_q5.SOURCE_ROW)
merged = raw_riasec.merge(raw_q5,on='SOURCE_ROW',how='outer',validate='one_to_one',indicator=True,sort=False)
display(merged['_merge'].value_counts().rename('Số dòng sau ghép').to_frame())
assert len(merged) == len(raw_riasec) == len(raw_q5) == 4910
assert merged['_merge'].eq('both').all()
merged = merged.drop(columns='_merge').set_index('SOURCE_ROW').loc[raw_riasec.SOURCE_ROW].reset_index()
assert merged.SOURCE_ROW.equals(raw_riasec.SOURCE_ROW)
assert merged[question_columns].equals(raw_riasec[question_columns])
assert merged[q5_columns].equals(raw_q5.set_index('SOURCE_ROW').loc[raw_riasec.SOURCE_ROW,q5_columns].reset_index(drop=True))
df_original = merged.copy(deep=True)
raw_survey = df_original[survey_columns].copy()
print('PASS: ghép một-một, không mất/thêm dòng, giữ thứ tự nguồn và 33 biến.')


PASS: ghép một-một, không mất/thêm dòng, giữ thứ tự nguồn và 33 biến.


            Số dòng sau ghép
_merge                      
both                    4910
left_only                  0
right_only                 0

## 4. Kiểm tra kiểu và miền giá trị
Kiểm tra cả kiểu ô: không chấp nhận chuỗi số hoặc boolean như đáp án. Float chỉ được chuyển nếu hữu hạn, nguyên và thuộc 1–5. Không điền thiếu, làm tròn hoặc sửa giá trị; nếu lỗi thì báo số lượng theo cột và dừng, không in phản hồi cá nhân.

In [3]:
type_rows = []
for col in survey_columns:
    values = raw_survey[col]
    numeric = pd.to_numeric(values,errors='coerce')
    is_numeric_cell = values.map(lambda value:isinstance(value,(int,float,np.integer,np.floating)) and not isinstance(value,(bool,np.bool_)))
    valid = is_numeric_cell & numeric.isin([1,2,3,4,5])
    type_rows.append({'Cột':col,'dtype gốc':str(values.dtype),'Thiếu':int(values.isna().sum()),
        'Sai kiểu hoặc ngoài miền':int((values.notna() & ~valid).sum()),
        'Min':numeric.min(),'Max':numeric.max(),'Số mức':numeric.nunique()})
type_check = pd.DataFrame(type_rows).set_index('Cột')
display(type_check)
if type_check['Thiếu'].sum() or type_check['Sai kiểu hoặc ngoài miền'].sum():
    raise RuntimeError('Có giá trị sai/thiếu; dừng để quyết định xử lý, không tự sửa.')
survey_int = raw_survey.astype('int64')
assert np.array_equal(survey_int.to_numpy(),raw_survey.to_numpy())
assert survey_int.isin(range(1,6)).all().all()
assert all(pd.api.types.is_integer_dtype(dtype) for dtype in survey_int.dtypes)
answers27 = survey_int[question_columns].copy()
preferences6 = survey_int[q5_columns].copy()
print('PASS: đủ 33 biến, không thiếu/sai kiểu/ngoài miền; chuyển int64 không đổi giá trị.')


PASS: đủ 33 biến, không thiếu/sai kiểu/ngoài miền; chuyển int64 không đổi giá trị.


          dtype gốc  Thiếu  Sai kiểu hoặc ngoài miền  Min  Max  Số mức
Cột                                                                   
R1            int64      0                         0    1    5       5
I1            int64      0                         0    1    5       5
A1            int64      0                         0    1    5       5
R2            int64      0                         0    1    5       5
I2            int64      0                         0    1    5       5
A2            int64      0                         0    1    5       5
S1            int64      0                         0    1    5       5
S2            int64      0                         0    1    5       5
R3            int64      0                         0    1    5       5
A3            int64      0                         0    1    5       5
S3            int64      0                         0    1    5       5
E1            int64      0                         0    1    5       5
R4    

## 5. Vector trùng và Q5 khác nhau
Group chỉ dựa trên đủ 27 câu, giữ nguyên thứ tự. Người tham gia khác nhau có thể trả lời giống X27 nhưng khác sở thích Q5. Đây không tự động là lỗi nhãn; không xóa/hợp nhất hoặc thay đổi Q5. Không hiển thị vector, SOURCE_ROW hoặc danh sách phản hồi cá nhân.

In [4]:
vector_groups = pd.Series(pd.factorize(pd.MultiIndex.from_frame(answers27),sort=False)[0],index=answers27.index,name='group_X27')
vector_sizes = vector_groups.value_counts()
duplicate_ids = vector_sizes.index[vector_sizes.gt(1)]
q5_distinct = preferences6.groupby(vector_groups).nunique()
different_q5_ids = q5_distinct.index[q5_distinct.gt(1).any(axis=1)]
duplicate_stats = pd.Series({
    'Vector X27 phân biệt':len(vector_sizes),
    'Dòng trùng thêm X27':int(answers27.duplicated().sum()),
    'Nhóm X27 trùng':len(duplicate_ids),
    'Dòng thuộc nhóm trùng':int(vector_sizes.loc[duplicate_ids].sum()),
    'Nhóm trùng lớn nhất':int(vector_sizes.max()),
    'Nhóm cùng X27 khác Q5':len(different_q5_ids),
    'Dòng thuộc nhóm cùng X27 khác Q5':int(vector_groups.isin(different_q5_ids).sum()),
    'Dòng trùng thêm đủ 33 câu':int(survey_int.duplicated().sum())})
reference00 = pd.Series([4460,450,57,507,188,54,501,239],index=duplicate_stats.index)
duplicate_comparison = pd.DataFrame({'Notebook 00':reference00,'Notebook 01':duplicate_stats})
duplicate_comparison['Chênh lệch'] = duplicate_stats-reference00
display(duplicate_comparison)
assert duplicate_comparison['Chênh lệch'].eq(0).all()
print('Phân bố kích thước các nhóm trùng (không hiển thị vector):')
display(vector_sizes.loc[duplicate_ids].value_counts().sort_index().rename_axis('Số dòng/nhóm').rename('Số nhóm').to_frame())
print('Kích thước nhóm lớn nhất:',vector_sizes.max(),'; số nhóm đạt kích thước này:',int(vector_sizes.eq(vector_sizes.max()).sum()))


Phân bố kích thước các nhóm trùng (không hiển thị vector):
Kích thước nhóm lớn nhất: 188 ; số nhóm đạt kích thước này: 1


                                  Notebook 00  Notebook 01  Chênh lệch
Vector X27 phân biệt                     4460         4460           0
Dòng trùng thêm X27                       450          450           0
Nhóm X27 trùng                             57           57           0
Dòng thuộc nhóm trùng                     507          507           0
Nhóm trùng lớn nhất                       188          188           0
Nhóm cùng X27 khác Q5                      54           54           0
Dòng thuộc nhóm cùng X27 khác Q5          501          501           0
Dòng trùng thêm đủ 33 câu                 239          239           0

              Số nhóm
Số dòng/nhóm         
2                  30
3                  11
4                   2
5                   5
6                   1
7                   2
14                  1
24                  1
38                  1
41                  1
56                  1
188                 1

Nhận xét: kết quả trùng khớp Notebook 00. 57 nhóm trùng và 54 nhóm có Q5 khác nhau được giữ nguyên. Khi chia Train/Test hoặc CV ở công đoạn sau, các vector X27 giống nhau phải nằm cùng một phía, kể cả khi Q5 khác nhau; không tạo lại group sau chọn đặc trưng.

## 6. Chất lượng phản hồi theo từng dòng
Quy định mô tả trước khi tính: variance ddof=0 bằng 0 tương ứng cùng một mức; **0 < variance ≤0,10** là ít biến thiên nhưng chưa hằng. Ngưỡng 0,10 là tiêu chí minh họa do nghiên cứu đề xuất, chưa có bằng chứng tâm trắc để dùng loại mẫu. Báo riêng cả nhóm variance ≤0,10 bao gồm các dòng phương sai bằng 0. Q5 không tham gia tạo cờ ít biến thiên hoặc quyết định xóa mẫu.


In [5]:
response_variance = answers27.var(axis=1,ddof=0)
straightline = answers27.nunique(axis=1).eq(1)
zero_variance = response_variance.eq(0)
assert straightline.equals(zero_variance)
assert int(straightline.sum()) == 347
straight_counts = answers27.loc[straightline,question_columns[0]].value_counts().reindex(range(1,6),fill_value=0)
straight_table = pd.DataFrame({'Số dòng':straight_counts,'Tỷ lệ toàn tập (%)':straight_counts/4910*100})
straight_table.index.name = 'Mức chung của cả 27 câu'
display(straight_table.round(4))
print('Tổng trả lời cùng mức:',int(straightline.sum()),'; tỷ lệ (%):',straightline.mean()*100)
low_variance_threshold = 0.10
low_nonzero = response_variance.gt(0) & response_variance.le(low_variance_threshold)
review_mask = response_variance.le(low_variance_threshold)
# Chỉ lưu cờ và tham chiếu trong bộ nhớ; không in danh sách SOURCE_ROW.
quality_flags = pd.DataFrame({'SOURCE_ROW':df_original.SOURCE_ROW.copy(),
    'variance_X27':response_variance,'cung_mot_muc':straightline,
    'bien_thien_thap_khac_0':low_nonzero,'can_xem_xet_bien_thien':review_mask})
flag_summary = pd.DataFrame({
    'Tiêu chí':['Variance = 0','0 < variance <= 0,10','Variance <= 0,10 (gồm variance=0)'],
    'Số dòng':[int(zero_variance.sum()),int(low_nonzero.sum()),int(review_mask.sum())]})
flag_summary['Tỷ lệ (%)'] = flag_summary['Số dòng']/4910*100
display(flag_summary.round(4))
display(response_variance.describe().rename('Phương sai phản hồi X27 (ddof=0)').to_frame().round(6))
assert len(quality_flags) == 4910
assert int(zero_variance.sum()+low_nonzero.sum()) == int(review_mask.sum())
print('Không áp dụng bất kỳ cờ nào để xóa/lọc mẫu; không dùng Q5 để làm đẹp hiệu năng.')


Tổng trả lời cùng mức: 347 ; tỷ lệ (%): 7.067209775967413
Không áp dụng bất kỳ cờ nào để xóa/lọc mẫu; không dùng Q5 để làm đẹp hiệu năng.


                         Số dòng  Tỷ lệ toàn tập (%)
Mức chung của cả 27 câu                             
1                             41              0.8350
2                             38              0.7739
3                            188              3.8289
4                             56              1.1405
5                             24              0.4888

                            Tiêu chí  Số dòng  Tỷ lệ (%)
0                       Variance = 0      347     7.0672
1               0 < variance <= 0,10      250     5.0916
2  Variance <= 0,10 (gồm variance=0)      597    12.1589

       Phương sai phản hồi X27 (ddof=0)
count                       4910.000000
mean                           0.764973
std                            0.629986
min                            0.000000
25%                            0.299040
50%                            0.620027
75%                            1.064472
max                            3.821674

Tiêu chí xem xét chất lượng gồm: kiểu/miền, thiếu giá trị, quan hệ ghép một-một, trùng vector và ít biến thiên. Ba nhóm đầu là kiểm tra tính toàn vẹn; trùng/ít biến thiên chỉ là dấu hiệu cần xem xét thêm về quy trình thu thập hoặc thời gian trả lời nếu có nguồn được phép. Chưa có bằng chứng để quy kết phản hồi không hợp lệ. Không loại 347 dòng cùng mức hoặc các dòng ít biến thiên khác.

## 7. Dữ liệu sạch trong bộ nhớ
Tách đúng X27, Q5 và SOURCE_ROW; không thêm metadata, nhóm/điểm suy ra hoặc nhãn ngành vào X27. DataFrame sạch gồm 33 câu và một cột truy vết, còn X27 chỉ có 27 câu. Không xuất file hoặc tạo nhãn từ điểm RIASEC.


In [6]:
df_clean = df_original[['SOURCE_ROW']].copy()
df_clean[survey_columns] = survey_int.copy()
X27 = df_clean[question_columns].copy()
Q5 = df_clean[q5_columns].copy()
SOURCE_ROW = df_clean['SOURCE_ROW'].copy()
assert df_clean.shape == (4910,34)
assert X27.shape == (4910,27) and Q5.shape == (4910,6)
assert list(X27.columns) == question_columns and list(Q5.columns) == q5_columns
assert not set(q5_columns+['SOURCE_ROW']).intersection(X27.columns)
assert SOURCE_ROW.is_unique and SOURCE_ROW.equals(df_original.SOURCE_ROW)
assert X27.index.equals(Q5.index) and X27.index.equals(SOURCE_ROW.index)
assert np.array_equal(X27.to_numpy(),df_original[question_columns].to_numpy())
assert np.array_equal(Q5.to_numpy(),df_original[q5_columns].to_numpy())
assert X27.notna().all().all() and Q5.notna().all().all()
assert X27.isin(range(1,6)).all().all() and Q5.isin(range(1,6)).all().all()
assert all(str(dtype)=='int64' for dtype in pd.concat([X27,Q5],axis=1).dtypes)
display(pd.DataFrame({'Đối tượng':['df_clean','X27','Q5','SOURCE_ROW'],
    'Số dòng':[len(df_clean),len(X27),len(Q5),len(SOURCE_ROW)],
    'Số cột':[df_clean.shape[1],X27.shape[1],Q5.shape[1],1],
    'Vai trò':['33 câu + cột truy vết','27 câu đầu vào','6 mức sở thích riêng','Truy vết, không là feature']}))
print('Thứ tự X27:',', '.join(X27.columns))
print('Các biến Q5:',', '.join(Q5.columns))


Thứ tự X27: R1, I1, A1, R2, I2, A2, S1, S2, R3, A3, S3, E1, R4, S4, E2, R5, I3, A4, E3, I4, A5, E4, C1, E5, C2, C3, C4
Các biến Q5: Q5_LikeRe, Q5_LikeIn, Q5_LikeAr, Q5_LikeSo, Q5_LikeEn, Q5_LikeCo


    Đối tượng  Số dòng  Số cột                     Vai trò
0    df_clean     4910      34       33 câu + cột truy vết
1         X27     4910      27              27 câu đầu vào
2          Q5     4910       6        6 mức sở thích riêng
3  SOURCE_ROW     4910       1  Truy vết, không là feature

## 8. Bảng trước/sau và kiểm tra bảo toàn
Không học tham số từ dữ liệu. Chỉ ghép theo khóa, kiểm tra và chuyển dtype bảo toàn giá trị. Không áp dụng quy tắc lọc, imputation, scaling, feature selection hoặc mô hình.

In [7]:
after_survey = pd.concat([X27,Q5],axis=1)
comparison = pd.DataFrame({
    'Chỉ số':['Số bản ghi','Số câu RIASEC','Số biến Q5','Ô thiếu (33 câu)',
        'Ô ngoài miền (33 câu)','Dòng bị loại','Vector X27 phân biệt',
        'Dòng trùng thêm X27','Nhóm vector X27 trùng','Dòng variance phản hồi = 0'],
    'Trước':[len(df_original),len(question_columns),len(q5_columns),int(raw_survey.isna().sum().sum()),
        int((~raw_survey.isin(range(1,6))).sum().sum()),0,
        len(raw_survey[question_columns].drop_duplicates()),int(raw_survey[question_columns].duplicated().sum()),
        len(duplicate_ids),int(raw_survey[question_columns].var(axis=1,ddof=0).eq(0).sum())],
    'Sau':[len(df_clean),X27.shape[1],Q5.shape[1],int(after_survey.isna().sum().sum()),
        int((~after_survey.isin(range(1,6))).sum().sum()),len(df_original)-len(df_clean),
        len(X27.drop_duplicates()),int(X27.duplicated().sum()),
        int(X27.value_counts().gt(1).sum()),int(X27.var(axis=1,ddof=0).eq(0).sum())]})
comparison['Chênh lệch'] = comparison['Sau']-comparison['Trước']
display(comparison)
assert comparison['Chênh lệch'].eq(0).all()
pd.testing.assert_frame_equal(raw_riasec,original_riasec)
pd.testing.assert_frame_equal(raw_q5,original_q5)
pd.testing.assert_frame_equal(raw_codebook,original_codebook)
pd.testing.assert_frame_equal(raw_info,original_info)
pd.testing.assert_frame_equal(df_original,merged)
sha_after = hashlib.sha256(workbook_path.read_bytes()).hexdigest()
assert sha_before == sha_after
print('SHA-256 trước:',sha_before)
print('SHA-256 sau:  ',sha_after)
print('PASS: giữ toàn bộ giá trị/dòng và thứ tự; dữ liệu gốc/workbook không đổi.')


SHA-256 trước: c8a9445edba02a37cf862a3b1eabffba910eb7eeaf1d48b4c6f8ac19fea296c8
SHA-256 sau:   c8a9445edba02a37cf862a3b1eabffba910eb7eeaf1d48b4c6f8ac19fea296c8
PASS: giữ toàn bộ giá trị/dòng và thứ tự; dữ liệu gốc/workbook không đổi.


                       Chỉ số  Trước   Sau  Chênh lệch
0                  Số bản ghi   4910  4910           0
1               Số câu RIASEC     27    27           0
2                  Số biến Q5      6     6           0
3            Ô thiếu (33 câu)      0     0           0
4       Ô ngoài miền (33 câu)      0     0           0
5                Dòng bị loại      0     0           0
6        Vector X27 phân biệt   4460  4460           0
7         Dòng trùng thêm X27    450   450           0
8       Nhóm vector X27 trùng     57    57           0
9  Dòng variance phản hồi = 0    347   347           0

## 9. Nhận xét và kết luận
Không phát hiện lỗi thiếu/ngoài miền hoặc quan hệ ghép. Làm sạch ở công đoạn này gồm xác minh cấu trúc, dtype, ghép một-một và chuẩn bị bản sao int64 bảo toàn giá trị. Giữ đủ 4.910 dòng, không gán ngành, tính điểm ngành phù hợp, tạo nhãn từ điểm RIASEC hoặc lọc dựa Q5.

Các cờ ít biến thiên và trùng vector không phải quyết định loại mẫu. Không tự thay Q5 khi cùng X27 có Q5 khác nhau. Các công đoạn chia tập/CV sau phải giữ group theo đủ 27 câu và chỉ fit phép biến đổi học tham số bên trong Train/training folds. Chưa chọn hướng A/B, chưa chia Train/Test hoặc huấn luyện ở notebook này.

Kiểm chứng: chạy tuần tự toàn bộ code cell bằng tiến trình Python 3.14 mới; outputs lưu trực tiếp trong notebook. Dùng phương án được cho phép theo lỗi quyền Jupyter Windows đã ghi nhận; chưa kiểm chứng qua giao diện/kernel Jupyter. Không xuất CSV/Excel/JSON/hình ảnh hoặc file phụ. Dừng sau Notebook 01, không tạo Notebook 02.


### Kết quả thực tế sau kiểm chứng
- **7/7 code cell** chạy thành công bằng tiến trình Python 3.14 mới. Ghép SOURCE_ROW một-một: **4.910 → 4.910** dòng; không mất hoặc thêm dòng.
- 33 biến hợp lệ, không thiếu/sai kiểu/ngoài miền. Chuyển int64 bảo toàn toàn bộ giá trị, không điền thiếu hoặc làm tròn. X27 **(4910,27)**, Q5 **(4910,6)**, SOURCE_ROW giữ riêng; df_clean **(4910,34)** chỉ tồn tại trong bộ nhớ.
- Trùng vector khớp Notebook 00: **4.460 vector phân biệt; 450 dòng trùng thêm; 57 nhóm/507 dòng thuộc nhóm trùng**. Nhóm lớn nhất **188 dòng**. **54 nhóm/501 dòng** cùng X27 có Q5 khác nhau; không gọi là lỗi nhãn, không sửa Q5 hoặc xóa/hợp nhất.
- Cùng mức cho cả 27 câu: mức 1 **41 dòng (0,8350%)**; mức 2 **38 (0,7739%)**; mức 3 **188 (3,8289%)**; mức 4 **56 (1,1405%)**; mức 5 **24 (0,4888%)**. Tổng **347 dòng (7,0672%)**.
- Có thêm **250 dòng (5,0916%)** với 0 < variance ≤0,10. Tổng cờ variance ≤0,10: **597 dòng (12,1589%)**. Ngưỡng là đề xuất mô tả, chưa có cơ sở để loại mẫu. Danh sách cờ có truy vết chỉ nằm trong bộ nhớ, không hiển thị SOURCE_ROW hoặc phản hồi cá nhân.
- **0 dòng bị loại**, không áp dụng quy tắc lọc dựa Q5. Bảng trước/sau không có chênh lệch. SHA-256 workbook giữ nguyên: `c8a9445edba02a37cf862a3b1eabffba910eb7eeaf1d48b4c6f8ac19fea296c8`.
- Đã kiểm tra hash của Notebook 00 RIASEC, pipeline F54, hai workbook, source website/kiểm thử và cấu hình liên quan không thay đổi trong quá trình chạy. Chưa kiểm chứng bằng kernel Jupyter. Dừng ở Notebook 01, không tạo Notebook 02.
